In [3]:
import gcode
import numpy as np

In [6]:
def init(**kwargs) -> gcode.GCode:
    """
    Initializes the machine with the given G-code settings.

    This function initializes a G-code program using the provided
    arguments. It sets the measurement units to millimeters (G21),
    absolute positioning mode (G90), and configures the machine's
    zero point at the K30 cross hairs (G92). Additionally, it sets
    motion velocities for the machine.

    Parameters
    ----------
    **kwargs : dict, optional
        Keyword arguments passed to the `gcode.GCode` constructor.
        These arguments define the G-code program settings.

    Returns
    -------
    gcode.GCode
        An initialized G-code program with the configured settings.

    Notes
    -----
    The function performs the following operations:
    1. Sets units to millimeters (G21).
    2. Sets absolute positioning (G90).
    3. Zeroes the machine at the K30 cross hairs (G92).
    4. Sets the motion velocity (G1 F1000, G0 F1000).
    5. Moves the machine to position (X=16.5, Y=1)
    6. Zeroes the coordinates (G92 X=0, Y=0).
    """
    prog = gcode.GCode.GCode(**kwargs)
    prog.G21()
    prog.G90()
    # Zero on the K30 cross hairs.
    prog.G92(X=0, Y=0)
    # Set motion velocity.
    prog.G1(F=1000)
    prog.G0(F=1000)
    # Offset for the K30 crosshairs.
    prog.G0(X=16.5, Y=1)
    # Re-zero.
    prog.G92(X=0, Y=0)
    # Laser Off
    prog.M5()

    return prog

In [7]:
feeds = [100, 250, 500, 750, 1000, 2000, 3000]
passes = [1, 2, 3, 4]

G00 X10 Y5   ; Rapid move to the starting point (10, 5)

G02 X10 Y5 I-5 J0   ; Draw a full circle clockwise

In [28]:
# Initial starting position.
X = 15
Y = 10
radius = 5
prog = init()
for pass_idx, pass_ in enumerate(passes):
    for _ in range(pass_):
        X = 15
        for feed_idx, feed in enumerate(feeds):
            prog.G1(X=X, Y=Y, S=0)
            prog.G1(X=X, Y=Y, F=feed)
            prog.M3(S=100)
            prog.G2(X=X, Y=Y, I=-5, J=0, F=feed)
            prog.M5()
            X += 2 * radius + 2.5

    X = 15
    Y += 2 * radius + 2.5

# Add Axes
prog += gcode.Line(points=np.array([[0, 0], [90, 0]]), feed=500, power=100)
prog += gcode.Line(points=np.array([[0, 0], [0, 52.5]]), feed=500, power=100)

prog.G0(X=0, Y=0)
prog.save("/tmp/circle_pass_cut.ngc")